# Interactive 3D Face Mapping

This notebook extracts the 3D topography of a face (X, Y, and Z depth) from a single photo using MediaPipe Face Landmarker, and renders it as an **interactive hologram** with Plotly, plus 2D projection views.

**Instructions (Google Colab):**
1. Run all cells in order (`Runtime > Run all`).
2. When prompted, upload a clear, front-facing photo (JPG/PNG).
3. In the interactive 3D plot, **drag to rotate**, **scroll to zoom**, and **shift-drag to pan**.

> Tip: use a well-lit, front-facing photo with the whole face visible for best results.


## 1. Install dependencies

In [ ]:
# Pinned, Colab-tested versions.
# mediapipe already pulls in a compatible numpy / opencv-contrib-python / matplotlib,
# so we don't pin those separately to avoid dependency conflicts.
!pip install -q "mediapipe==0.10.35" "plotly==5.24.1" "nbformat==5.10.4"


## 2. Imports and model setup

In [ ]:
import os
import math
import urllib.request

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# --- Ensure the face landmarker model exists ---
MODEL_PATH = "face_landmarker.task"
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        MODEL_PATH,
    )

# --- Ensure the face mesh connection definitions exist ---
try:
    import face_mesh_connections as fmc
except ImportError:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/google/mediapipe/master/mediapipe/python/solutions/face_mesh_connections.py",
        "face_mesh_connections.py",
    )
    import face_mesh_connections as fmc

# FULL DETAILED FACIAL FEATURES (Eyes, Eyebrows, Lips, Face Oval, Nose)
CONNECTIONS = [
    fmc.FACEMESH_LIPS,
    fmc.FACEMESH_LEFT_EYE,
    fmc.FACEMESH_LEFT_EYEBROW,
    fmc.FACEMESH_RIGHT_EYE,
    fmc.FACEMESH_RIGHT_EYEBROW,
    fmc.FACEMESH_FACE_OVAL,
    fmc.FACEMESH_NOSE,
]

print("Setup complete.")


## 3. Upload your photo

`tkinter` file dialogs don't work in a hosted Colab runtime (there's no local desktop), so this uses Colab's built-in upload widget instead. If you're running this locally in Jupyter instead of Colab, a fallback file-picker is provided below.

In [ ]:
USER_IMG_PATH = None

try:
    # Running in Google Colab
    from google.colab import files

    print("Please select your face photo for 3D mapping...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No image selected.")
    USER_IMG_PATH = next(iter(uploaded.keys()))

except ImportError:
    # Fallback for local Jupyter (desktop) environments
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.attributes("-topmost", True)
        root.withdraw()
        USER_IMG_PATH = filedialog.askopenfilename(
            title="Select Photo",
            filetypes=[("Images", "*.png *.jpg *.jpeg")],
        )
        root.destroy()
        if not USER_IMG_PATH:
            raise ValueError("No image selected.")
    except Exception as e:
        raise RuntimeError(
            "No GUI file dialog available. Set USER_IMG_PATH manually to a path, e.g.\n"
            "USER_IMG_PATH = '/content/my_photo.jpg'"
        ) from e

print(f"Selected: {USER_IMG_PATH}")


## 4. Extract 3D landmarks and build the point cloud

In [ ]:
def extract_3d_landmarks(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read {image_path}")
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        num_faces=1,
        min_face_detection_confidence=0.5,
    )

    with vision.FaceLandmarker.create_from_options(options) as detector:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)
        if not result.face_landmarks:
            raise ValueError("No face detected in the image.")
        return result.face_landmarks[0]


landmarks = extract_3d_landmarks(USER_IMG_PATH)

# --- Calculate proportional depth scaling ---
x_vals = [lm.x for lm in landmarks]
z_vals = [lm.z for lm in landmarks]
x_range = np.max(x_vals) - np.min(x_vals)
z_range = np.max(z_vals) - np.min(z_vals)

TARGET_Z_RANGE = x_range * 0.60
Z_MULTIPLIER = TARGET_Z_RANGE / z_range if z_range > 0 else 1.0

print(f"Dynamically scaled Z-depth by {Z_MULTIPLIER:.2f}x for perfect proportions.")

# --- Prepare data for Plotly 3D ---
pts_3d = np.zeros((len(landmarks), 3))
for i, lm in enumerate(landmarks):
    pts_3d[i] = [lm.x, -lm.z * Z_MULTIPLIER, -lm.y]

# --- SYNTHETIC FOREHEAD GENERATION ---
top_oval_indices = [21, 54, 103, 67, 109, 10, 338, 297, 332, 284, 251]
p_chin = pts_3d[152]
p_top = pts_3d[10]
face_height = math.hypot(p_chin[0] - p_top[0], p_chin[2] - p_top[2])
center_x = p_top[0]

for idx in top_oval_indices:
    p = pts_3d[idx]
    sx = center_x + (p[0] - center_x) * 0.85
    sy = p[1] - (face_height * 0.1)   # push backward (Y is depth)
    sz = p[2] + (face_height * 0.22)  # push upward (Z is height)
    pts_3d[idx] = [sx, sy, sz]

print(f"Extracted {len(landmarks)} landmarks.")


## 5. Interactive 3D hologram

Drag to rotate, scroll to zoom, shift-drag to pan.

In [ ]:
# Build wireframe line segments from the mesh connections
edge_x, edge_y, edge_z = [], [], []
for connection_set in CONNECTIONS:
    for start_idx, end_idx in connection_set:
        p1, p2 = pts_3d[start_idx], pts_3d[end_idx]
        edge_x += [p1[0], p2[0], None]
        edge_y += [p1[1], p2[1], None]
        edge_z += [p1[2], p2[2], None]

fig = go.Figure()

# Wireframe mesh
fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode="lines",
    line=dict(color="#00e5ff", width=3),
    hoverinfo="skip",
    name="Face mesh",
))

# Landmark points
fig.add_trace(go.Scatter3d(
    x=pts_3d[:, 0], y=pts_3d[:, 1], z=pts_3d[:, 2],
    mode="markers",
    marker=dict(size=2, color="#ffffff", opacity=0.6),
    hoverinfo="skip",
    name="Landmarks",
))

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0a0a0a",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        camera=dict(eye=dict(x=0, y=-2.0, z=0.3)),
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    title="Interactive 3D Face Hologram",
    showlegend=False,
    height=700,
)

fig.show()


## 6. 2D projections (front photo overlay + side profile)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor("#0a0a0a")

# --- 1. Overlay on Original 2D Photo (Front Plane) ---
img = mpimg.imread(USER_IMG_PATH)
ax1.imshow(img)
image_h, image_w = img.shape[0], img.shape[1]

for connection_set in CONNECTIONS:
    for start_idx, end_idx in connection_set:
        x1 = landmarks[start_idx].x * image_w
        y1 = landmarks[start_idx].y * image_h
        x2 = landmarks[end_idx].x * image_w
        y2 = landmarks[end_idx].y * image_h
        ax1.plot([x1, x2], [y1, y2], color="#00e5ff", linewidth=2)

ax1.axis("off")
ax1.set_title("2D Photo Projection", color="white", fontsize=16)

# --- 2. Side Profile 2D Projection (Depth vs Height) ---
ax2.set_facecolor("#121212")
for connection_set in CONNECTIONS:
    for start_idx, end_idx in connection_set:
        z1 = -landmarks[start_idx].z * Z_MULTIPLIER
        y1 = -landmarks[start_idx].y
        z2 = -landmarks[end_idx].z * Z_MULTIPLIER
        y2 = -landmarks[end_idx].y
        ax2.plot([z1, z2], [y1, y2], color="#00e5ff", linewidth=2)

ax2.set_aspect("equal")
ax2.axis("off")
ax2.set_title("2D Side Profile Projection", color="white", fontsize=16)

plt.tight_layout()
plt.show()


## 7. Original photo vs. mapped wireframe

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor("#0a0a0a")

img = mpimg.imread(USER_IMG_PATH)
image_h, image_w = img.shape[0], img.shape[1]

# --- 1. Original Image ---
ax1.imshow(img)
ax1.axis("off")
ax1.set_title("Original Photo", color="white", fontsize=18, pad=15)

# --- 2. Mapped Image (wireframe over photo) ---
ax2.imshow(img)
for connection_set in CONNECTIONS:
    for start_idx, end_idx in connection_set:
        x1 = landmarks[start_idx].x * image_w
        y1 = landmarks[start_idx].y * image_h
        x2 = landmarks[end_idx].x * image_w
        y2 = landmarks[end_idx].y * image_h
        ax2.plot([x1, x2], [y1, y2], color="#00e5ff", linewidth=3, alpha=1.0)

ax2.axis("off")
ax2.set_title("3D Mapping on 2D Plane", color="white", fontsize=18, pad=15)

plt.tight_layout()
plt.show()
